# Project 56 — Local CLI Command Planner Agent

**Suggest shell commands with explanation and human approval before execution.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangGraph |
| Tools | subprocess (sandboxed) |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to build a **command-planning agent** that translates English to shell commands
2. How to implement **risk assessment** for CLI commands
3. How to use **human-in-the-loop approval** before executing commands
4. How to build a **LangGraph workflow** with conditional routing
5. How to sandbox command execution for safety

## 2 · Why This Project Matters

CLI is powerful but intimidating. A command planner agent helps users by translating
natural-language requests into shell commands while explaining what each command does
and what risks it carries. The approval gate ensures no destructive command runs
without explicit consent.

## 3 · Environment Setup

**Prerequisites:**
- Ollama running with `qwen3:8b`

In [ ]:
# !pip install -q langchain langchain-ollama langgraph pydantic

## 4 · Imports and Configuration

In [ ]:
import json
import platform
import subprocess
from typing import TypedDict

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)

OS_INFO = f'{platform.system()} {platform.release()}'
SHELL = 'powershell' if platform.system() == 'Windows' else 'bash'
print(f'LLM ready: {MODEL}')
print(f'OS: {OS_INFO}, Shell: {SHELL}')

## 5 · Define the Command Planning Logic

The LLM generates commands and assesses their risk level (safe, moderate, dangerous).

In [ ]:
def plan_command(request: str) -> dict:
    """Generate a shell command plan from a natural-language request."""
    prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a CLI expert on {os} using {shell}. Given a user request, respond '
         'with a JSON object (no markdown fencing):\n'
         '{{"command": "the shell command", '
         '"explanation": "what it does in plain English", '
         '"risk_level": "safe|moderate|dangerous", '
         '"risk_reason": "why this risk level", '
         '"alternatives": ["alternative command if any"]}}\n\n'
         'Rules:\n'
         '- safe: read-only or informational commands (ls, cat, echo, dir, Get-ChildItem)\n'
         '- moderate: creates or modifies files/configs\n'
         '- dangerous: deletes files, modifies system settings, installs software\n'
         '- Always prefer the safest approach'),
        ('human', '{request}'),
    ])
    chain = prompt | llm | StrOutputParser()
    raw = chain.invoke({'request': request, 'os': OS_INFO, 'shell': SHELL})

    # Try to parse JSON from response
    try:
        # Find JSON in response
        start = raw.find('{')
        end = raw.rfind('}') + 1
        if start >= 0 and end > start:
            plan = json.loads(raw[start:end])
        else:
            plan = {'command': raw, 'explanation': 'Could not parse structured response',
                    'risk_level': 'moderate', 'risk_reason': 'Unparsed', 'alternatives': []}
    except json.JSONDecodeError:
        plan = {'command': raw, 'explanation': 'Could not parse structured response',
                'risk_level': 'moderate', 'risk_reason': 'Unparsed', 'alternatives': []}

    return plan


# Test
test_plan = plan_command('List all Python files in the current directory')
print(json.dumps(test_plan, indent=2))

## 6 · Build the LangGraph Approval Workflow

```
Plan Command → Assess Risk → [Safe? Execute] [Moderate/Dangerous? Approval Gate] → Execute or Reject
```

In [ ]:
from langgraph.graph import StateGraph, END


class CommandState(TypedDict):
    request: str
    plan: dict
    approved: bool
    auto_approve_safe: bool
    result: str


def plan_node(state: CommandState) -> CommandState:
    plan = plan_command(state['request'])
    return {**state, 'plan': plan}


def approval_node(state: CommandState) -> CommandState:
    plan = state['plan']
    risk = plan.get('risk_level', 'moderate')

    # Auto-approve safe commands if configured
    if risk == 'safe' and state.get('auto_approve_safe', False):
        return {**state, 'approved': True}

    # For demo, we simulate approval based on risk
    if risk == 'dangerous':
        return {**state, 'approved': False,
                'result': f'BLOCKED: Dangerous command rejected.\n'
                          f'Command: {plan.get("command", "")}\n'
                          f'Reason: {plan.get("risk_reason", "")}'}

    # Moderate commands get approved for demo
    return {**state, 'approved': True}


def execute_node(state: CommandState) -> CommandState:
    if not state.get('approved', False):
        return state  # Already has rejection message

    command = state['plan'].get('command', '')

    # Allowlist: only execute safe read-only commands
    safe_prefixes = ['dir', 'ls', 'echo', 'python --version', 'Get-ChildItem',
                     'Get-Date', 'hostname', 'whoami', 'pwd', 'Get-Location']
    is_safe = any(command.strip().lower().startswith(p.lower()) for p in safe_prefixes)

    if not is_safe:
        return {**state, 'result': f'DRY-RUN (not in allowlist): {command}\n'
                                    f'Explanation: {state["plan"].get("explanation", "")}'}

    # Execute safe command
    try:
        result = subprocess.run(
            command, shell=True, capture_output=True, text=True, timeout=10
        )
        output = result.stdout.strip() or result.stderr.strip() or '(no output)'
        return {**state, 'result': f'EXECUTED: {command}\nOutput:\n{output[:500]}'}
    except subprocess.TimeoutExpired:
        return {**state, 'result': f'TIMEOUT: Command took too long: {command}'}
    except Exception as e:
        return {**state, 'result': f'ERROR: {e}'}


# Build graph
workflow = StateGraph(CommandState)
workflow.add_node('plan', plan_node)
workflow.add_node('approve', approval_node)
workflow.add_node('execute', execute_node)

workflow.set_entry_point('plan')
workflow.add_edge('plan', 'approve')
workflow.add_edge('approve', 'execute')
workflow.add_edge('execute', END)

app = workflow.compile()
print('CLI Command Planner workflow ready.')

## 7 · Test the Agent

In [ ]:
def run_cli_agent(request: str, auto_approve_safe: bool = True) -> dict:
    state = app.invoke({
        'request': request,
        'plan': {},
        'approved': False,
        'auto_approve_safe': auto_approve_safe,
        'result': '',
    })
    print(f'\nRequest: {request}')
    print(f'Command: {state["plan"].get("command", "N/A")}')
    print(f'Risk: {state["plan"].get("risk_level", "N/A")}')
    print(f'Explanation: {state["plan"].get("explanation", "N/A")}')
    print(f'Result: {state["result"]}')
    print('-' * 60)
    return state


# Safe command - should auto-execute
s1 = run_cli_agent('Show the current date and time')

In [ ]:
s2 = run_cli_agent('List files in the current directory')

In [ ]:
s3 = run_cli_agent('Show which version of Python is installed')

In [ ]:
# Moderate command - approved but dry-run (not in allowlist)
s4 = run_cli_agent('Create a new folder called test_output')

In [ ]:
# Dangerous command - should be blocked
s5 = run_cli_agent('Delete all files in the temp directory')

## 8 · Command History and Audit Trail

Track all commands for auditing.

In [ ]:
history = [s1, s2, s3, s4, s5]

print('COMMAND AUDIT TRAIL')
print('=' * 60)
for i, s in enumerate(history, 1):
    plan = s['plan']
    print(f'{i}. [{plan.get("risk_level", "?").upper()}] {plan.get("command", "N/A")[:60]}')
    approved = 'YES' if s.get('approved') else 'NO'
    print(f'   Approved: {approved}')

## 9 · Save Results

In [ ]:
output = []
for s in history:
    output.append({
        'request': s['request'],
        'plan': s['plan'],
        'approved': s.get('approved', False),
        'result': s['result'],
    })
with open('cli_planner_results.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f'Saved {len(output)} command results')

## 10 · Limitations

- **Allowlist execution** — only whitelisted commands actually run
- **OS-specific** — commands differ between Windows/Linux/macOS
- **No multi-step** — each request maps to one command (no piping/chaining)
- **JSON parsing** can fail with small models
- Risk assessment is LLM-based and not always accurate

## 11 · How to Improve

1. **Multi-step commands**: support piped commands and scripts
2. **Interactive approval**: real input() prompt for human approval
3. **Command history**: suggest similar past commands
4. **OS detection**: auto-adapt commands for the current OS
5. **Docker sandbox**: execute untrusted commands in containers

## 12 · Mini Challenge

1. Add support for multi-command plans (e.g., 'create folder and move files into it')
2. Implement a `--dry-run` mode that shows what would happen without executing
3. Add a `learn_command` tool that explains a command the user doesn't understand

## 13 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| NL-to-CLI | LLM translates English to shell commands |
| Risk assessment | Classify commands as safe/moderate/dangerous |
| Approval workflow | LangGraph with conditional execution |
| Safe execution | Allowlist + dry-run for untrusted commands |
| Audit trail | Log all commands with approval status |